# 05 · Predict and write the submission

Same idea as the organisers' notebook 02:
1. Train the final model on **all training recordings** of all 6 subjects (the rules allow
   sub-05/06's *training* files).
2. Save the trained encoders to `weights/` (sharing weights is required).
3. **Format check on sub-01**: build the ranked list exactly like the real submission and score
   it with sub-01's answers. It must match notebook 04.
4. Write `submission/sub-05_ranked_candidate_list.npy` and `sub-06_…`: 102 × 102 integers. Row *i*
   = the *i*-th recording of `tst/sub-0X_tst.npy` in its **original order**. Each row lists the
   video ids 1001–1102, most likely first.

The sub-05/06 test recordings are used **only here**, once, to produce the predictions.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # repo root, so `mindreading` imports work
import numpy as np
import matplotlib.pyplot as plt
from mindreading.config import DATA_DIR, RESULTS_DIR, WEIGHTS_DIR, SUBMISSION_DIR
import warnings
warnings.filterwarnings('ignore')            # keep library warnings (with install paths) out of the outputs
from mindreading.config import REPO


def rel(p):
    """Path relative to the repo, so outputs never show anyone's home folder."""
    try:
        return Path(p).relative_to(REPO).as_posix()
    except ValueError:
        return f'{Path(p).name} (outside the repo, set via MINDREADING_DATA)'


print('data folder:', rel(DATA_DIR))

data folder: data


In [2]:
import joblib
from mindreading.model import Config, Model, predict_scores
from mindreading.ranking import scores_to_ranked_ids, check_ranked, evaluate, save_submission

cfg = Config()
model = Model(cfg).fit_shared()
members = {s: {**model.fit_subject(s), **model.shared} for s in (1, 5, 6)}
WEIGHTS_DIR.mkdir(exist_ok=True)
weights = {'config': cfg.__dict__,
           'shared': {k: {'mu': m.mu, 'coef': m.ridge.coef_.astype(np.float32), 'intercept': m.ridge.intercept_}
                      for k, m in model.shared.items()},
           'subject': {s: {'mu': m['subject'].mu, 'coef': m['subject'].ridge.coef_.astype(np.float32),
                           'intercept': m['subject'].ridge.intercept_} for s, m in members.items()}}
joblib.dump(weights, WEIGHTS_DIR / 'ensemble_encoders.joblib', compress=3)
print('saved', rel(WEIGHTS_DIR / 'ensemble_encoders.joblib'))

saved weights/ensemble_encoders.joblib


In [3]:
r1 = check_ranked(scores_to_ranked_ids(predict_scores(model, 1, members[1])))
labels1 = np.load(DATA_DIR / 'tst_stim' / 'sub-01_tst_stim.npy')
print('format check on sub-01:', r1.shape, r1.dtype, 'first row starts', r1[0, :5])
print('score with sub-01 answers:', evaluate(r1, labels1))

format check on sub-01: (102, 102) int64 first row starts [1077 1035 1030 1078 1023]
score with sub-01 answers: {'top1': 0.19607843137254902, 'top5': 0.5}


In [4]:
SUBMISSION_DIR.mkdir(exist_ok=True)
for sub in (5, 6):
    ranked = check_ranked(scores_to_ranked_ids(predict_scores(model, sub, members[sub])))
    path = save_submission(ranked, f'sub-{sub:02d}', SUBMISSION_DIR)
    back = check_ranked(np.load(path))
    print(rel(path), back.shape, back.dtype, 'row 0 top-5:', back[0, :5])

submission/sub-05_ranked_candidate_list.npy (102, 102) int64 row 0 top-5: [1075 1051 1082 1079 1018]


submission/sub-06_ranked_candidate_list.npy (102, 102) int64 row 0 top-5: [1007 1079 1086 1043 1015]
